# Google Colab(Pro recommended): T4 GPU setup

1. **Runtime → Change runtime type**
   - **Hardware accelerator**: `T4 GPU`
   - Click **Connect** on the top-right
2. Run the cells **top to bottom** once per new session.

This notebook checks the driver, confirms PyTorch sees the GPU, installs a **CUDA-enabled JAX** stack (needed for this repo’s `jax` / `jaxlib` pins), and installs `requirements.txt`.

In [ ]:
# Driver and GPU model (expect NVIDIA T4 on many GPU runtimes)
!nvidia-smi

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])

import torch

# Colab usually ships CUDA PyTorch; only reinstall if the runtime has no GPU build.
if not torch.cuda.is_available():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "torch",
            "torchvision",
            "torchaudio",
            "--index-url",
            "https://download.pytorch.org/whl/cu124",
        ]
    )
    raise RuntimeError(
        "Installed CUDA PyTorch. Use Runtime → Restart session, then run all cells again."
    )

assert torch.cuda.is_available(), "No CUDA GPU visible. Set Runtime → GPU and re-run."
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f"PyTorch {torch.__version__} | GPU: {name} | capability {cap}")
x = torch.randn(4096, 4096, device="cuda")
y = x @ x
print("GEMM on GPU:", float(y[0, 0].cpu()))

In [ ]:
# JAX with CUDA (matches typical Colab CUDA 12.x). Removes CPU-only jax if present.
import subprocess, sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "uninstall", "-y", "jax", "jaxlib"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "jax[cuda12]",
        "jaxlib",
        "-f",
        "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ]
)

import jax
print("JAX", jax.__version__, "devices:", jax.devices())
x = jax.numpy.ones((4096, 4096))
y = x @ x
y.block_until_ready()
print("JAX matmul ok on", y.device)

## Project setup for this repo (`graph-rerank-pipeline`)

- Clone the repo and install dependencies.
- Then set runtime environment variables in Colab. Your local `.env` is **not** included when you clone from GitHub, so defaults are used unless you set values in a cell.
- For API keys (`cohere`, etc.), use Colab **Secrets** (key icon) or `os.environ` — do not commit keys.

In [ ]:
# Clone and install (edit the URL only if needed)
!git clone https://github.com/puppy06/graph-rerank-pipeline.git
%cd /content/graph-rerank-pipeline
!pip install -q -r requirements.txt

In [ ]:
# Set runtime env vars for this Colab session.
# Important: cloned repos do not include your local .env file.
import os

os.environ["USE_LOCAL_MODEL"] = "true"
os.environ["LOCAL_LLM_MODEL_ID"] = "Qwen/Qwen2.5-3B-Instruct"
os.environ["LOCAL_EMBED_MODEL_ID"] = "BAAI/bge-m3"
os.environ["LOCAL_DEVICE"] = "cuda"

print("USE_LOCAL_MODEL=", os.environ["USE_LOCAL_MODEL"])
print("LOCAL_LLM_MODEL_ID=", os.environ["LOCAL_LLM_MODEL_ID"])
print("LOCAL_EMBED_MODEL_ID=", os.environ["LOCAL_EMBED_MODEL_ID"])
print("LOCAL_DEVICE=", os.environ["LOCAL_DEVICE"])

In [ ]:
# Ingest and then run retrieval-only query (no LLM generation)
!python scripts/rag.py ingest --data-dir corpus --reset
!python scripts/langgraph_rag.py --query "Compare NVIDIA and AMD Q4 gross margins." --skip-generate